# Кластеризация сообщений из чата поддержки

Ход анализа по шагам. Вся логика — в модулях [`src/`](src/), описание задачи и результатов — в [README](README.md), формат входных данных — в [data/README.md](data/README.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_messages, load_intents
from src.embeddings import embed_texts
from src.clustering import NOISE, reduce_dimensions, cluster_points, describe_clustering, grid_search, top_examples
from src.intent_filter import score_against_intents
from src.visualization import plot_clusters_3d

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Загрузка данных

In [ ]:
messages = load_messages(DATA_DIR / "messages.csv")
print(f"Сообщений: {len(messages)}")
messages.head()

## 2. Эмбеддинги

Каждое сообщение — вектор размерности 1024 от модели multilingual-e5-large.

In [ ]:
embeddings = embed_texts(messages["text_clean"].tolist())
embeddings.shape

## 3. Подбор параметров

Разведочный шаг: как число кластеров и доля шума зависят от параметров UMAP и HDBSCAN. Подходящими считаем варианты с 50–200 кластерами и шумом меньше 20% (критерий под выборку ~31 тыс. сообщений).

In [ ]:
configs = [
    {"n_neighbors": 15, "min_cluster_size": 15, "min_samples": 3, "epsilon": 0.05},
    {"n_neighbors": 15, "min_cluster_size": 20, "min_samples": 5, "epsilon": 0.10},
    {"n_neighbors": 30, "min_cluster_size": 15, "min_samples": 3, "epsilon": 0.05},
    {"n_neighbors": 30, "min_cluster_size": 20, "min_samples": 5, "epsilon": 0.10},
    {"n_neighbors": 30, "min_cluster_size": 30, "min_samples": 5, "epsilon": 0.10},
    {"n_neighbors": 30, "min_cluster_size": 30, "min_samples": 5, "epsilon": 0.15},
    {"n_neighbors": 50, "min_cluster_size": 20, "min_samples": 5, "epsilon": 0.10},
    {"n_neighbors": 50, "min_cluster_size": 30, "min_samples": 5, "epsilon": 0.15},
]
search = grid_search(embeddings, configs)

search["suitable"] = search["n_clusters"].between(50, 200) & (search["noise_share"] < 0.2)
search.sort_values(["suitable", "noise_share"], ascending=[False, True])

## 4. Кластеризация

UMAP понижает размерность до 50, HDBSCAN кластеризует. Сообщения, не попавшие ни в один кластер, получают метку шума (`-1`).

In [ ]:
points_50d = reduce_dimensions(embeddings, n_components=50, n_neighbors=15)
labels, probabilities = cluster_points(points_50d, min_cluster_size=10, min_samples=3, epsilon=0.12)

messages["cluster"] = labels
messages["cluster_prob"] = probabilities

stats = describe_clustering(labels)
pd.Series(stats)

Распределение кластеров по размеру:

In [ ]:
sizes = messages.loc[messages["cluster"] != NOISE, "cluster"].value_counts()
pd.cut(sizes, bins=[0, 10, 20, 50, 100, 300, 1000, np.inf]).value_counts().sort_index()

### Типичные фразы кластеров

10 сообщений, ближайших к центроиду кластера. По ним кластеры размечаются вручную. Полный список — в `outputs/cluster_examples.csv`, ниже — 20 самых крупных кластеров.

In [ ]:
examples = top_examples(messages["text_clean"], embeddings, labels, n=10)
examples.to_csv(OUTPUT_DIR / "cluster_examples.csv", index=False)

for cluster_id, group in list(examples.groupby("cluster", sort=False))[:20]:
    print(f"\n=== Кластер {cluster_id} (n={group['cluster_size'].iloc[0]}) ===")
    for text in group["text"]:
        print("  -", text[:150])

## 5. Визуализация

Отдельная проекция UMAP в 3D, только для картинки: кластеры берутся из шага 4. При наведении на точку виден текст сообщения.

In [ ]:
points_3d = reduce_dimensions(embeddings, n_components=3, n_neighbors=10)

title = f"{stats['n_clusters']} кластеров, шум {stats['noise_share']:.1%}"
fig = plot_clusters_3d(points_3d, labels, messages["text_clean"].str[:100], title=title)
fig.write_html(str(OUTPUT_DIR / "clusters_3d.html"))
fig.show()

## 6. Итоговая таблица

In [ ]:
result = messages[["text_id", "text_clean", "cluster", "freq"]]
result.to_csv(OUTPUT_DIR / "clusters.csv", index=False)
result.head(20)

## 7. Поиск новых формулировок для интентов

Берём кластеры целевой темы (открытие расчётного счёта) и сравниваем их сообщения с примерами из существующих интентов: по словам (TF-IDF по леммам) и по смыслу (эмбеддинги). Сообщения, у которых средняя близость ниже порога, не похожи ни на один пример — это кандидаты в новые примеры для интентов.

In [ ]:
# Номера кластеров выбраны вручную по типичным фразам из шага 4.
# На других данных номера будут другими.
TARGET_CLUSTERS = [124, 186, 225, 199, 194, 298, 193, 131, 240, 236, 133, 79, 103, 2, 0, 52, 121, 239, 141]
THRESHOLD = 0.6

intents = load_intents(DATA_DIR / "intents.txt")
intent_embeddings = embed_texts(intents)

in_target = messages["cluster"].isin(TARGET_CLUSTERS)
scored = score_against_intents(messages[in_target], embeddings[in_target.to_numpy()], intents, intent_embeddings)
scored[["sim_tfidf", "sim_emb", "sim_combined"]].describe()

Сколько сообщений остаётся при разных способах фильтрации:

In [ ]:
pd.Series({
    "TF-IDF < 0.3": (scored["sim_tfidf"] < 0.3).sum(),
    "эмбеддинги < 0.89": (scored["sim_emb"] < 0.89).sum(),
    f"среднее < {THRESHOLD}": (scored["sim_combined"] < THRESHOLD).sum(),
}, name="новых формулировок")

In [ ]:
new_examples = scored[scored["sim_combined"] < THRESHOLD].sort_values("freq", ascending=False)
new_examples.to_csv(OUTPUT_DIR / "new_intent_examples.csv", index=False)

print(f"Кандидатов в новые примеры: {len(new_examples)} из {len(scored)}")
new_examples[["text_clean", "cluster", "freq", "sim_tfidf", "sim_emb", "sim_combined", "closest_intent"]].head(50)

## 8. Шум

Сообщения, не попавшие ни в один кластер, — самые частые сверху.

In [ ]:
noise = messages[messages["cluster"] == NOISE].sort_values("freq", ascending=False)
noise.to_csv(OUTPUT_DIR / "noise.csv", index=False)

print(f"Шум: {len(noise)} сообщений ({len(noise) / len(messages):.1%})")
noise[["text_clean", "freq"]].head(50)